In [3]:
import sys; sys.path.append("../py")
from seoul_api import seoul_get, seoul_all
import os
import pandas as pd

print(os.environ.get("SEOUL_API_KEY", "키 없음")[:6], "...")

r = seoul_get("bikeList", 1, 5)
print(r["code"], r["msg"], r["total"])
if r["code"] is None:
    print(r.get("raw"))

785945 ...
INFO-000 정상 처리되었습니다. 5


In [4]:
import datetime as dt
today = dt.date.today()
probe = [today - dt.timedelta(days=k) for k in (1, 2, 3, 7, 14, 30, 60, 90)]
probe += [dt.date(2026, 6, 30), dt.date(2026, 1, 15), dt.date(2025, 6, 15)]

q1 = pd.DataFrame([
    {"date": d, **{k: v for k, v in seoul_get("tbCycleFailureReport", 1, 5, d.strftime("%Y%m%d")).items() if k != "rows"}}
    for d in probe])
q1

,date,total,code,msg
0,2026-09-28,981,INFO-000,정상 처리되었습니다.
1,2026-09-27,1347,INFO-000,정상 처리되었습니다.
2,2026-09-26,1646,INFO-000,정상 처리되었습니다.
3,2026-09-22,3157,INFO-000,정상 처리되었습니다.
4,2026-09-15,6768,INFO-000,정상 처리되었습니다.
5,2026-08-30,13851,INFO-000,정상 처리되었습니다.
6,2026-07-31,10487,INFO-000,정상 처리되었습니다.
7,2026-07-01,8346,INFO-000,정상 처리되었습니다.
8,2026-06-30,8465,INFO-000,정상 처리되었습니다.
9,2026-01-15,5201,INFO-000,정상 처리되었습니다.


In [6]:
d = seoul_all("tbCycleFailureReport", "20260701")
print(len(d)); print(d.columns.tolist())

# 날짜처럼 보이는 컬럼을 찾아 최소·최대를 출력
for c in d.columns:
    v = d[c].astype(str)
    if v.str.len().median() >= 8:
        s = pd.to_datetime(v, errors="coerce", format="mixed")
        if s.notna().mean() > 0.9:
            print(c, s.min(), "~", s.max())
d.head(3)

8346
['bikeNo', 'regDttm', 'mlangComCdName', 'START_INDEX', 'END_INDEX', 'RNUM']
regDttm 2026-07-01 00:19:34 ~ 2026-07-31 23:48:03


,bikeNo,regDttm,mlangComCdName,START_INDEX,END_INDEX,RNUM
0,SPB-67517,2026-07-01 00:19:34,기타,0,0,1
1,SPB-49509,2026-07-01 00:33:30,타이어,0,0,2
2,SPB-49509,2026-07-01 00:33:30,체인,0,0,3


In [7]:
d = seoul_all("tbCycleFailureReport", "20260701").drop(columns=["START_INDEX", "END_INDEX", "RNUM"], errors="ignore")
d["reg_ts"] = pd.to_datetime(d["regDttm"])
d["date"] = d["reg_ts"].dt.date

rows    = d.groupby("date").size()                                          # 유형 건수(행)
reports = d.drop_duplicates(["bikeNo", "regDttm"]).groupby("date").size()   # 신고 건수(고유)
daily = pd.DataFrame({"rows": rows, "reports": reports})
daily["types_per_report"] = (daily["rows"] / daily["reports"]).round(2)
print(daily.describe().round(1))
d["mlangComCdName"].value_counts()

        rows  reports  types_per_report
count   31.0     31.0              31.0
mean   269.2    236.2               1.1
std     76.1     65.4               0.0
min    150.0    133.0               1.1
25%    206.0    180.0               1.1
50%    244.0    221.0               1.1
75%    341.0    300.0               1.2
max    394.0    343.0               1.2


mlangComCdName
기타      2722
타이어     2009
체인      1300
안장      1072
페달       728
단말기      515
Name: count, dtype: int64

In [8]:
probe2 = [today - dt.timedelta(days=k) for k in (1, 7, 30, 90, 180)]
probe2 += [dt.date(2026, 6, 15), dt.date(2025, 6, 15), dt.date(2024, 6, 15)]
q2 = pd.DataFrame([{"date": d, "total": seoul_get("tbCycleRentData", 1, 5, d.strftime("%Y-%m-%d"), "08")["total"]}
                   for d in probe2])
q2

,date,total
0,2026-09-28,13933
1,2026-09-22,15558
2,2026-08-30,3458
3,2026-07-01,14506
4,2026-04-02,13247
5,2026-06-15,13381
6,2025-06-15,3709
7,2024-06-15,4630


In [9]:
h = seoul_all("tbCycleRentData", (today - dt.timedelta(days=1)).strftime("%Y-%m-%d"), "08")
print(len(h)); print(h.columns.tolist()); h.head()
h["USR_CLS_CD"].value_counts(), h["BIKE_SE_CD"].value_counts()

13933
['BIKE_ID', 'RENT_DT', 'RENT_ID', 'RENT_NM', 'RENT_HOLD', 'RTN_DT', 'RTN_ID', 'RTN_NM', 'RTN_HOLD', 'USE_MIN', 'USE_DST', 'USR_CLS_CD', 'BIRTH_YEAR', 'RENT_STATION_ID', 'RETURN_STATION_ID', 'BIKE_SE_CD', 'START_INDEX', 'END_INDEX', 'RNUM', 'SEX_CD']


(USR_CLS_CD
 USR_001    13913
 USR_003       17
 USR_002        3
 Name: count, dtype: int64,
 BIKE_SE_CD
 일반자전거    13407
 새싹자전거      526
 Name: count, dtype: int64)

In [10]:
# 보존 범위: 몇 년 전까지 나오는가 (모두 6월 15일 08시)
rows3 = []
for y in (2023, 2022, 2021, 2020, 2019, 2018):
    d0 = dt.date(y, 6, 15)
    r = seoul_get("tbCycleRentData", 1, 5, d0.strftime("%Y-%m-%d"), "08")
    rows3.append({"date": d0, "weekday": d0.strftime("%a"), "total": r["total"], "code": r["code"]})
pd.DataFrame(rows3)

,date,weekday,total,code
0,2023-06-15,Thu,17693,INFO-000
1,2022-06-15,Wed,4326,INFO-000
2,2021-06-15,Tue,2538,INFO-000
3,2020-06-15,Mon,7147,INFO-000
4,2019-06-15,Sat,2251,INFO-000
5,2018-06-15,Fri,3395,INFO-000


In [11]:
# 받은 한 시간이 정말 그 날짜 08시인지, 값 형식은 어떤지
print(h["RENT_DT"].min(), "~", h["RENT_DT"].max())
print(pd.to_numeric(h["USE_DST"], errors="coerce").describe().round(1))   # 단위 m인지(중앙값 1,000~3,000 근처면 m)
h[["BIKE_ID", "RENT_DT", "RTN_DT", "RENT_ID", "RENT_NM", "RENT_STATION_ID", "USE_MIN", "USE_DST"]].head(3)

2026-09-28 08:00:00 ~ 2026-09-28 08:59:59
count    13933.0
mean      1644.9
std       1875.3
min          0.0
25%        648.1
50%       1084.0
75%       1954.2
max      34086.1
Name: USE_DST, dtype: float64


,BIKE_ID,RENT_DT,RTN_DT,RENT_ID,RENT_NM,RENT_STATION_ID,USE_MIN,USE_DST
0,SPB-57881,2026-09-28 08:00:13,2026-09-28 08:00:23,01841,가산동 주민센터,ST-1142,0,0.10
1,SPB-40124,2026-09-28 08:00:33,2026-09-28 08:00:46,02745,수명초등학교,ST-2412,0,0.00
2,SPB-36415,2026-09-28 08:00:20,2026-09-28 08:00:48,00903,은평뉴타운 아이파크,ST-451,0,0.00


In [13]:
# 헛대여가 얼마나 있는지 알아보기
for c in ["USE_MIN", "USE_DST"]:
    h[c] = pd.to_numeric(h[c], errors="coerce")
sham = (h["USE_MIN"] <= 3) & (h["RENT_ID"] == h["RTN_ID"]) & (h["USE_DST"] <= 300)
print(sham.sum(), f"{sham.mean():.1%}")
h.loc[sham, "BIKE_ID"].value_counts().head(5)

386 2.8%


BIKE_ID
SPB-71293    5
SPB-64405    5
SPB-46587    4
SPB-48602    4
SPB-33822    3
Name: count, dtype: int64

In [14]:
f_recent = seoul_all("tbCycleFailureReport", (today - dt.timedelta(days=7)).strftime("%Y%m%d"))
rep = set(f_recent["bikeNo"])
cnt = h.loc[sham, "BIKE_ID"].value_counts()
groups = {"헛대여 2회+": set(cnt[cnt >= 2].index),
          "헛대여 1회": set(cnt[cnt == 1].index),
          "정상 이용만": set(h["BIKE_ID"]) - set(cnt.index)}
pd.DataFrame([{"그룹": g, "자전거 수": len(b), "신고 비율": round(len(b & rep) / max(len(b), 1), 3)}
              for g, b in groups.items()])

,그룹,자전거 수,신고 비율
0,헛대여 2회+,35,0.400
1,헛대여 1회,300,0.103
2,정상 이용만,9471,0.056


In [16]:
import codecs
from pathlib import Path
ROOT = Path("..")

def sniff(path, n=200_000):
    """파일 앞부분으로 형식·인코딩을 추정한다"""
    with open(path, "rb") as fh:
        raw = fh.read(n)
    if raw[:2] == b"PK":
        return "zip/xlsx"
    if raw[:2] in (b"\xff\xfe", b"\xfe\xff"):
        return "utf-16"
    if raw.lstrip()[:1] == b"<":
        return "html"
    for enc in ("utf-8-sig", "cp949"):
        try:
            codecs.getincrementaldecoder(enc)().decode(raw, final=False)
            return enc
        except UnicodeDecodeError:
            pass
    return "mixed"

def _best_guess(path, n=5_000_000):
    with open(path, "rb") as fh:
        raw = fh.read(n)
    return min(("utf-8-sig", "cp949"), key=lambda e: raw.decode(e, errors="replace").count("\ufffd"))

def _copy(src, dst, enc, errors):
    n_bad = 0
    with open(src, encoding=enc, errors=errors, newline="") as fi, \
         open(dst, "w", encoding="utf-8", newline="") as fo:
        for line in fi:
            if errors == "replace":
                n_bad += line.count("\ufffd")
            fo.write(line)
    return n_bad

def to_utf8(src, dst):
    """원본은 그대로 두고 UTF-8 사본을 만든다. 반환: (인코딩, 깨진 글자 수)"""
    enc = sniff(src)
    if enc in ("zip/xlsx", "html"):
        return enc + " → 변환 안 함", None
    Path(dst).parent.mkdir(parents=True, exist_ok=True)
    if enc != "mixed":
        try:
            return enc, _copy(src, dst, enc, "strict")
        except UnicodeDecodeError:
            pass
    enc = _best_guess(src)
    return enc + " (깨진 글자는 �로 대체)", _copy(src, dst, enc, "replace")

rows = []
for f in sorted(p for p in (ROOT / "data/raw/failure").iterdir() if p.suffix.lower() == ".csv"):
    try:
        enc, n_bad = to_utf8(f, ROOT / "data/interim/failure_utf8" / f.name)
    except Exception as e:
        enc, n_bad = f"오류: {e}", None
    rows.append({"file": f.name, "MB": round(f.stat().st_size / 1e6, 1), "encoding": enc, "깨진 글자": n_bad})
conv_log = pd.DataFrame(rows)
conv_log

,file,MB,encoding,깨진 글자
0,서울시 공공자전거 고장신고 내역_2020.11~2021.01.csv,0.2,cp949,0.0
1,서울시 공공자전거 고장신고 내역_21.02-21.06.csv,1.4,cp949,0.0
2,서울시 공공자전거 고장신고 내역_21.7-21.12.csv,2.7,cp949,0.0
3,서울시 공공자전거 고장신고 내역_22.01-06.csv,2.5,cp949,0.0
4,서울시 공공자전거 고장신고 내역_22.07-12.csv,2.8,zip/xlsx → 변환 안 함,NaN
5,서울시 공공자전거 고장신고 내역_23.1-6.csv,2.8,cp949,0.0
6,서울시 공공자전거 고장신고 내역_23.11-12.csv,0.9,cp949,0.0
7,서울시 공공자전거 고장신고 내역_23.7-10.csv,2.8,cp949,0.0
8,서울시 공공자전거 고장신고 내역_2401-2406.csv,3.2,cp949,0.0
9,서울시 공공자전거 고장신고 내역_2407-2412.csv,3.6,cp949,0.0


In [17]:
import io, zipfile
for f in conv_log.loc[conv_log["encoding"].str.startswith("zip/xlsx"), "file"]:
    src = ROOT / "data/raw/failure" / f
    names = zipfile.ZipFile(src).namelist()
    if "[Content_Types].xml" in names:                      # 엑셀 파일(이름만 .csv)
        sheets = pd.read_excel(io.BytesIO(src.read_bytes()), sheet_name=None, dtype=str, engine="openpyxl")
        print(f, {k: v.shape for k, v in sheets.items()})
        x = pd.concat(sheets.values(), ignore_index=True)
        x.to_csv(ROOT / "data/interim/failure_utf8" / f, index=False, encoding="utf-8")
    else:                                                    # 진짜 압축 파일
        print(f, "압축 안의 파일:", names)

서울시 공공자전거 고장신고 내역_22.07-12.csv {'Sheet1': (105550, 3)}


In [18]:
# 0-6·0-8에서 쓸 고장신고 파일: 목록을 보고 2026년 상반기 파일 번호를 고른다
fail_files = sorted((ROOT / "data/interim/failure_utf8").glob("*.csv"))
for i, f in enumerate(fail_files):
    print(i, f.name)

0 서울시 공공자전거 고장신고 내역_2020.11~2021.01.csv
1 서울시 공공자전거 고장신고 내역_21.02-21.06.csv
2 서울시 공공자전거 고장신고 내역_21.7-21.12.csv
3 서울시 공공자전거 고장신고 내역_22.01-06.csv
4 서울시 공공자전거 고장신고 내역_22.07-12.csv
5 서울시 공공자전거 고장신고 내역_23.1-6.csv
6 서울시 공공자전거 고장신고 내역_23.11-12.csv
7 서울시 공공자전거 고장신고 내역_23.7-10.csv
8 서울시 공공자전거 고장신고 내역_2401-2406.csv
9 서울시 공공자전거 고장신고 내역_2407-2412.csv
10 서울시 공공자전거 고장신고 내역_2501-2506.csv
11 서울시 공공자전거 고장신고 내역_2507-2512.csv
12 서울시 공공자전거 고장신고 내역_2601-2606.csv


In [20]:
FAIL_2026H1 = fail_files[12]          # ← 위 목록에서 2026년 상반기 파일 번호로 바꾼다
print(FAIL_2026H1.name)

서울시 공공자전거 고장신고 내역_2601-2606.csv


In [21]:
for f in sorted((ROOT / "data/raw/rent_history").iterdir()):
    print(f.name, f"{f.stat().st_size / 1e6:,.0f} MB")

서울특별시 공공자전거 대여이력 정보_2015.zip 4 MB
서울특별시 공공자전거 대여이력 정보_2016.zip 56 MB
서울특별시 공공자전거 대여이력 정보_2017.zip 202 MB
서울특별시 공공자전거 대여이력 정보_2018.zip 441 MB
서울특별시 공공자전거 대여이력 정보_2019.zip 854 MB
서울특별시 공공자전거 대여이력 정보_2020.zip 1,022 MB
서울특별시 공공자전거 대여이력 정보_2021.zip 1,474 MB
서울특별시 공공자전거 대여이력 정보_2022.zip 1,906 MB
서울특별시 공공자전거 대여이력 정보_2023.zip 2,105 MB
서울특별시 공공자전거 대여이력 정보_2024.zip 2,069 MB
서울특별시 공공자전거 대여이력 정보_2025.zip 1,779 MB
서울특별시 공공자전거 대여이력 정보_2601.csv 287 MB
서울특별시 공공자전거 대여이력 정보_2602.csv 314 MB
서울특별시 공공자전거 대여이력 정보_2603.csv 513 MB
서울특별시 공공자전거 대여이력 정보_2604.csv 690 MB
서울특별시 공공자전거 대여이력 정보_2605.csv 706 MB
서울특별시 공공자전거 대여이력 정보_2606.csv 734 MB


In [22]:
src = ROOT / "data/raw/rent_history" / "서울특별시 공공자전거 대여이력 정보_2601.csv"
RENT_202601 = ROOT / "data/interim/rent_csv/rent_202601_utf8.csv"
print(to_utf8(src, RENT_202601))

('cp949', 0)


In [23]:
def peek(path):
    x = pd.read_csv(path, nrows=5, dtype=str, encoding="utf-8")
    return {"file": Path(path).name, "n_col": x.shape[1], "cols": " | ".join(x.columns),
            "first_row": " | ".join(map(str, x.iloc[0].tolist()))}

files = list((ROOT / "data/interim").rglob("*.csv"))
pd.DataFrame([peek(f) for f in files])

,file,n_col,cols,first_row
0,서울시 공공자전거 고장신고 내역_23.1-6.csv,3,자전거번호 | 등록일시 | 구분,SPB-62819 | 2023-01-01 0:01 | 안장
1,서울시 공공자전거 고장신고 내역_23.11-12.csv,3,자전거번호 | 등록일시 | 고장구분,SPB-67589 | 2023-11-01 00:00:50 | 타이어
2,서울시 공공자전거 고장신고 내역_2407-2412.csv,3,자전거번호 | 등록일시 | 구분,SPB-54796 | 2024-07-01 00:03:39 | 기타
3,서울시 공공자전거 고장신고 내역_2601-2606.csv,3,자전거번호 | 등록일시 | 구분,SPB-74348 | 2026-01-01 00:19:33 | 타이어
4,서울시 공공자전거 고장신고 내역_22.07-12.csv,3,자전거번호 | 등록일시 | 고장구분,SPB-30229 | 2022-07-01 00:44:33 | 페달
5,서울시 공공자전거 고장신고 내역_2020.11~2021.01.csv,3,자전거번호 | 등록일시 | 고장구분,SPB-37924 | 2020-11-01 0:10 | 기타
6,서울시 공공자전거 고장신고 내역_23.7-10.csv,3,자전거번호 | 등록일시 | 고장구분,SPB-49493 | 2023-07-01 00:02:28 | 기타
7,서울시 공공자전거 고장신고 내역_22.01-06.csv,3,자전거번호 | 등록일시 | 고장구분,SPB-36049 | 2022-01-01 0:45 | 기타
8,서울시 공공자전거 고장신고 내역_2401-2406.csv,3,자전거번호 | 등록일시 | 구분,SPB-53779 | 2024-01-01 01:11:39 | 안장
9,서울시 공공자전거 고장신고 내역_2501-2506.csv,3,자전거번호 | 등록일시 | 구분,SPB-70626 | 2025-01-01 00:12:27 | 기타


In [25]:
print(peek(RENT_202601)["cols"])

자전거번호 | 대여일시 | 대여 대여소번호 | 대여 대여소명 | 대여거치대 | 반납일시 | 반납대여소번호 | 반납대여소명 | 반납거치대 | 이용시간(분) | 이용거리(M) | 생년 | 성별 | 이용자종류 | 대여대여소ID | 반납대여소ID


In [26]:
import codecs, zipfile

def _ok(raw, enc):
    try:
        codecs.getincrementaldecoder(enc)().decode(raw, final=False)
        return True
    except UnicodeDecodeError:
        return False

def _head(raw):
    enc = next((e for e in ("utf-8-sig", "cp949") if _ok(raw, e)), "?")
    lines = raw.decode("cp949" if enc == "?" else enc, errors="replace").splitlines()
    return enc, (lines[0] if lines else ""), (lines[1] if len(lines) > 1 else "")

def peek_rent(folder):
    """ZIP은 풀지 않고 안의 파일 앞부분만 읽는다"""
    rows = []
    for p in sorted(Path(folder).iterdir()):
        if p.suffix.lower() == ".zip":
            with zipfile.ZipFile(p) as zf:
                for m in zf.infolist():
                    if m.is_dir():
                        continue
                    name = m.filename
                    if not m.flag_bits & 0x800:            # 윈도우에서 만든 ZIP은 한글 이름이 cp949
                        try: name = name.encode("cp437").decode("cp949")
                        except (UnicodeEncodeError, UnicodeDecodeError): pass
                    if not name.lower().endswith(".csv"):
                        rows.append({"source": p.name, "file": name, "MB": round(m.file_size / 1e6), "enc": "csv 아님"})
                        continue
                    with zf.open(m) as fh:
                        enc, header, row1 = _head(fh.read(20_000))
                    rows.append({"source": p.name, "file": name, "MB": round(m.file_size / 1e6),
                                 "enc": enc, "n_col": header.count(",") + 1, "header": header, "row1": row1})
        elif p.suffix.lower() == ".csv":
            with open(p, "rb") as fh:
                enc, header, row1 = _head(fh.read(20_000))
            rows.append({"source": p.name, "file": p.name, "MB": round(p.stat().st_size / 1e6),
                         "enc": enc, "n_col": header.count(",") + 1, "header": header, "row1": row1})
    return pd.DataFrame(rows)

rent_peek = peek_rent(ROOT / "data/raw/rent_history")
print(rent_peek.groupby("source")["MB"].sum())
rent_peek[["source", "file", "MB", "enc", "n_col"]]

source
서울특별시 공공자전거 대여이력 정보_2015.zip      20
서울특별시 공공자전거 대여이력 정보_2016.zip     297
서울특별시 공공자전거 대여이력 정보_2017.zip     936
서울특별시 공공자전거 대여이력 정보_2018.zip    1885
서울특별시 공공자전거 대여이력 정보_2019.zip    3554
서울특별시 공공자전거 대여이력 정보_2020.zip    4397
서울특별시 공공자전거 대여이력 정보_2021.zip    6000
서울특별시 공공자전거 대여이력 정보_2022.zip    7686
서울특별시 공공자전거 대여이력 정보_2023.zip    8438
서울특별시 공공자전거 대여이력 정보_2024.zip    8245
서울특별시 공공자전거 대여이력 정보_2025.zip    7029
서울특별시 공공자전거 대여이력 정보_2601.csv     287
서울특별시 공공자전거 대여이력 정보_2602.csv     314
서울특별시 공공자전거 대여이력 정보_2603.csv     513
서울특별시 공공자전거 대여이력 정보_2604.csv     690
서울특별시 공공자전거 대여이력 정보_2605.csv     706
서울특별시 공공자전거 대여이력 정보_2606.csv     734
Name: MB, dtype: int64


,source,file,MB,enc,n_col
0,서울특별시 공공자전거 대여이력 정보_2015.zip,서울특별시 공공자전거 대여이력 정보_1512.csv,4,cp949,17
1,서울특별시 공공자전거 대여이력 정보_2015.zip,서울특별시 공공자전거 대여이력 정보_1511.csv,7,cp949,17
2,서울특별시 공공자전거 대여이력 정보_2015.zip,서울특별시 공공자전거 대여이력 정보_1510.csv,7,cp949,17
3,서울특별시 공공자전거 대여이력 정보_2015.zip,서울특별시 공공자전거 대여이력 정보_1509.csv,2,cp949,17
4,서울특별시 공공자전거 대여이력 정보_2016.zip,서울특별시 공공자전거 대여이력 정보_1605.csv,18,cp949,17
...,...,...,...,...,...
125,서울특별시 공공자전거 대여이력 정보_2602.csv,서울특별시 공공자전거 대여이력 정보_2602.csv,314,cp949,16
126,서울특별시 공공자전거 대여이력 정보_2603.csv,서울특별시 공공자전거 대여이력 정보_2603.csv,513,cp949,16
127,서울특별시 공공자전거 대여이력 정보_2604.csv,서울특별시 공공자전거 대여이력 정보_2604.csv,690,cp949,16
128,서울특별시 공공자전거 대여이력 정보_2605.csv,서울특별시 공공자전거 대여이력 정보_2605.csv,706,cp949,16


In [27]:
for h, g in rent_peek.dropna(subset=["header"]).groupby("header", sort=False):
    print(f"[{len(g)}개 파일] {g['file'].iloc[0]} ~ {g['file'].iloc[-1]}")
    print("   ", h)
    print("    예:", g["row1"].iloc[0])

[124개 파일] 서울특별시 공공자전거 대여이력 정보_1512.csv ~ 서울특별시 공공자전거 대여이력 정보_2506.csv
    "자전거번호","대여일시","대여 대여소번호","대여 대여소명","대여거치대","반납일시","반납대여소번호","반납대여소명","반납거치대","이용시간(분)","이용거리(M)","생년","성별","이용자종류","대여대여소ID","반납대여소ID","자전거구분"
    예: "SPB-00546","2015-12-01 00:05:58","00216","삼부아파트1동 앞","10","2015-12-01 00:10:49","00210","IFC몰","9","4","1140.00","1992","M","내국인","ST-65","ST-56","LCD자전거"
[6개 파일] 서울특별시 공공자전거 대여이력 정보_2601.csv ~ 서울특별시 공공자전거 대여이력 정보_2606.csv
    "자전거번호","대여일시","대여 대여소번호","대여 대여소명","대여거치대","반납일시","반납대여소번호","반납대여소명","반납거치대","이용시간(분)","이용거리(M)","생년","성별","이용자종류","대여대여소ID","반납대여소ID"
    예: "SPB-65030","2026-01-01 00:00:13","01729","도봉한신아파트 버스정류장","0","2026-01-01 00:02:26","01729","도봉한신아파트 버스정류장","0","2","0.00","","M","내국인","ST-1461","ST-1461"


In [28]:
import duckdb
(ROOT / "data/db").mkdir(parents=True, exist_ok=True)
con = duckdb.connect(str(ROOT / "data/db/phase0.duckdb"))

con.execute(f"""CREATE OR REPLACE TABLE rent AS
  SELECT * FROM read_csv('{RENT_202601.resolve().as_posix()}', header = true, all_varchar = true)""")
con.execute(f"""CREATE OR REPLACE TABLE fail AS
  SELECT * FROM read_csv('{FAIL_2026H1.resolve().as_posix()}', header = true, all_varchar = true)""")

con.sql("DESCRIBE rent").show()        # 실제 컬럼 이름 확인
con.sql("DESCRIBE fail").show()

┌─────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │ column_type │  null   │   key   │ default │  extra  │
│     varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ 자전거번호      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ 대여일시        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ 대여 대여소번호 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ 대여 대여소명   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ 대여거치대      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ 반납일시        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ 반납대여소번호  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ 반납대여소명    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ 반납거치대      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ 이용시간(분)    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ 이용거리(M)     │ VA

In [29]:
con.sql("""
WITH r AS (
  SELECT DISTINCT CAST(regexp_extract("자전거번호", '[0-9]+') AS INTEGER) AS bike FROM rent
),
f AS (
  SELECT CAST(regexp_extract("자전거번호", '[0-9]+') AS INTEGER) AS bike FROM fail
  WHERE "등록일시" >= '2026-01-15' AND "등록일시" < '2026-02-01'
)
SELECT count(*) AS n_reports,
       avg(CASE WHEN r.bike IS NOT NULL THEN 1.0 ELSE 0 END) AS match_rate
FROM f LEFT JOIN r USING (bike)
""").show()

┌───────────┬────────────────────┐
│ n_reports │     match_rate     │
│   int64   │       double       │
├───────────┼────────────────────┤
│      3076 │ 0.9986996098829649 │
└───────────┴────────────────────┘



In [30]:
con.sql("""
WITH r AS (SELECT DISTINCT CAST(regexp_extract("자전거번호", '[0-9]+') AS INTEGER) AS bike FROM rent),
     f AS (SELECT DISTINCT CAST(regexp_extract("자전거번호", '[0-9]+') AS INTEGER) AS bike, "자전거번호" AS raw
           FROM fail WHERE "등록일시" >= '2026-01-01' AND "등록일시" < '2026-02-01')
SELECT (SELECT count(*) FROM r) AS bikes_rented,
       (SELECT count(*) FROM f) AS bikes_reported,
       (SELECT count(*) FROM r JOIN f USING (bike)) AS both_
""").show()

con.sql("""
SELECT f."자전거번호" FROM fail f
LEFT JOIN (SELECT DISTINCT CAST(regexp_extract("자전거번호", '[0-9]+') AS INTEGER) AS bike FROM rent) r
  ON CAST(regexp_extract(f."자전거번호", '[0-9]+') AS INTEGER) = r.bike
WHERE r.bike IS NULL AND f."등록일시" >= '2026-01-15' AND f."등록일시" < '2026-02-01'
LIMIT 20
""").show()

┌──────────────┬────────────────┬───────┐
│ bikes_rented │ bikes_reported │ both_ │
│    int64     │     int64      │ int64 │
├──────────────┼────────────────┼───────┤
│        36290 │           4461 │  4452 │
└──────────────┴────────────────┴───────┘

┌────────────┐
│ 자전거번호 │
│  varchar   │
├────────────┤
│ SPB-34468  │
│ SPB-64292  │
│ SPB-53622  │
│ SPB-63582  │
└────────────┘



In [31]:
api = seoul_all("tbCycleFailureReport", "20260601")
api = api[pd.to_datetime(api["regDttm"]).dt.strftime("%Y-%m") == "2026-06"]
fl = pd.read_csv(FAIL_2026H1, dtype=str)
fl = fl[pd.to_datetime(fl["등록일시"], format="mixed").dt.strftime("%Y-%m") == "2026-06"]
print("API 6월 행:", len(api), " 파일 6월 행:", len(fl))
key = ["bikeNo", "regDttm", "kind"]
m = (api.rename(columns={"mlangComCdName": "kind"})[key]
       .merge(fl.rename(columns={"자전거번호": "bikeNo", "등록일시": "regDttm", "구분": "kind"})[key],
              on=key, how="outer", indicator=True))
m["_merge"].value_counts()

API 6월 행: 11548  파일 6월 행: 11548


_merge
both          11548
left_only         0
right_only        0
Name: count, dtype: int64

In [32]:
con.sql("""
SELECT count(*) AS n,
  avg(CASE WHEN TRY_CAST("이용거리(M)" AS DOUBLE) IS NULL THEN 1.0 ELSE 0 END) AS na_rate,
  avg(CASE WHEN TRY_CAST("이용거리(M)" AS DOUBLE) = 0 THEN 1.0 ELSE 0 END)     AS zero_rate,
  quantile_cont(TRY_CAST("이용거리(M)" AS DOUBLE), 0.5)  AS median_m,
  quantile_cont(TRY_CAST("이용거리(M)" AS DOUBLE), 0.99) AS p99_m
FROM rent
""").show()

┌─────────┬─────────┬─────────────────────┬──────────┬───────────────────┐
│    n    │ na_rate │      zero_rate      │ median_m │       p99_m       │
│  int64  │ double  │       double        │  double  │      double       │
├─────────┼─────────┼─────────────────────┼──────────┼───────────────────┤
│ 1640582 │     0.0 │ 0.03534233582960193 │  1141.71 │ 10448.50609999999 │
└─────────┴─────────┴─────────────────────┴──────────┴───────────────────┘



In [33]:
con.sql("""
WITH t AS (
  SELECT TRY_CAST("이용거리(M)" AS DOUBLE) / 1000 AS km,
         TRY_CAST("이용시간(분)" AS DOUBLE)      AS mins,
         "대여 대여소번호" = "반납대여소번호"      AS same_st
  FROM rent
)
SELECT
  quantile_cont(km / (mins / 60), 0.5)  FILTER (WHERE mins >= 1 AND km > 0) AS speed_median,
  quantile_cont(km / (mins / 60), 0.99) FILTER (WHERE mins >= 1 AND km > 0) AS speed_p99,
  avg(CASE WHEN mins >= 1 AND km / (mins / 60) > 35 THEN 1.0 ELSE 0 END)    AS over35_rate,
  avg(CASE WHEN km = 0 AND same_st     THEN 1.0 ELSE 0 END)                  AS zero_same_st,
  avg(CASE WHEN km = 0 AND NOT same_st THEN 1.0 ELSE 0 END)                  AS zero_diff_st
FROM t
""").show()

┌──────────────────┬───────────┬───────────────────────┬──────────────────────┬──────────────────────┐
│   speed_median   │ speed_p99 │      over35_rate      │     zero_same_st     │     zero_diff_st     │
│      double      │  double   │        double         │        double        │        double        │
├──────────────────┼───────────┼───────────────────────┼──────────────────────┼──────────────────────┤
│ 8.68478181818182 │    19.182 │ 0.0006138065637682237 │ 0.024743658043304143 │ 0.009702654301948943 │
└──────────────────┴───────────┴───────────────────────┴──────────────────────┴──────────────────────┘



In [34]:
f = pd.read_csv(FAIL_2026H1, nrows=20, dtype=str)
f.iloc[:, :3]

,자전거번호,등록일시,구분
0,SPB-74348,2026-01-01 00:19:33,타이어
1,SPB-74469,2026-01-01 00:56:39,체인
2,SPB-43287,2026-01-01 01:14:12,기타
3,SPB-62249,2026-01-01 01:51:58,체인
4,SPB-36526,2026-01-01 01:52:09,기타
5,SPB-73566,2026-01-01 01:53:39,안장
6,SPB-73566,2026-01-01 01:53:39,기타
7,SPB-68914,2026-01-01 02:10:28,단말기
8,SPB-68914,2026-01-01 02:10:28,기타
9,SPB-45957,2026-01-01 03:21:09,타이어


In [35]:
for f in fail_files:
    d = pd.read_csv(f, dtype=str)
    t = d["등록일시"].str.contains(":", na=False)
    sec = d["등록일시"].str.count(":").eq(2)
    print(f"{f.name[-22:]:>24} {len(d):>8,}행  시각 있음 {t.mean():>5.0%}  초까지 {sec.mean():>5.0%}")

  내역_2020.11~2021.01.csv    6,749행  시각 있음  100%  초까지    0%
  장신고 내역_21.02-21.06.csv   53,843행  시각 있음    0%  초까지    0%
  고장신고 내역_21.7-21.12.csv   95,689행  시각 있음    0%  초까지    0%
  거 고장신고 내역_22.01-06.csv   74,924행  시각 있음  100%  초까지    0%
  거 고장신고 내역_22.07-12.csv  105,550행  시각 있음  100%  초까지  100%
  자전거 고장신고 내역_23.1-6.csv   83,734행  시각 있음  100%  초까지    0%
  거 고장신고 내역_23.11-12.csv   21,851행  시각 있음  100%  초까지  100%
  전거 고장신고 내역_23.7-10.csv   67,328행  시각 있음  100%  초까지  100%
   고장신고 내역_2401-2406.csv   76,464행  시각 있음  100%  초까지  100%
   고장신고 내역_2407-2412.csv   85,382행  시각 있음  100%  초까지  100%
   고장신고 내역_2501-2506.csv   53,574행  시각 있음  100%  초까지  100%
   고장신고 내역_2507-2512.csv   59,305행  시각 있음  100%  초까지  100%
   고장신고 내역_2601-2606.csv   52,612행  시각 있음  100%  초까지  100%


In [36]:
import requests                      # 노트북에서는 따로 불러 둔다(seoul_api.py 안의 import는 노트북에 안 보임)
r = requests.get(
    "http://apis.data.go.kr/1360000/AsosHourlyInfoService/getWthrDataList",
    params=dict(serviceKey=os.environ["DATA_GO_KR_KEY"], dataType="JSON", dataCd="ASOS", dateCd="HR",
                startDt="20260901", startHh="00", endDt="20260901", endHh="23",
                stnIds="108", numOfRows=24, pageNo=1),
    timeout=30)
js = r.json()
js["response"]["header"]["resultCode"], len(js["response"]["body"]["items"]["item"])   # ('00', 24)

('00', 24)

In [37]:
r = requests.get("http://apis.data.go.kr/1360000/AsosHourlyInfoService/getWthrDataList",
    params=dict(serviceKey=os.environ["DATA_GO_KR_KEY"], dataType="JSON", dataCd="ASOS", dateCd="HR",
                startDt="20210615", startHh="05", endDt="20210615", endHh="10",
                stnIds="108", numOfRows=10, pageNo=1), timeout=30)
w = pd.DataFrame(r.json()["response"]["body"]["items"]["item"])
w[["tm", "ta", "rn", "ws"]]          # 시각, 기온, 강수량(mm, 빈칸=0), 풍속

,tm,ta,rn,ws
0,2021-06-15 05:00,21.7,0.0,1.4
1,2021-06-15 06:00,21.5,0.0,1.2
2,2021-06-15 07:00,21.6,,1.0
3,2021-06-15 08:00,21.9,0.0,2.6
4,2021-06-15 09:00,22.0,0.2,1.9
5,2021-06-15 10:00,22.0,0.2,2.0


In [38]:
import io, zipfile
z = ROOT / "data/raw/rent_history/서울특별시 공공자전거 대여이력 정보_2021.zip"
with zipfile.ZipFile(z) as zf:
    def nm(m): return m.filename if m.flag_bits & 0x800 else m.filename.encode("cp437").decode("cp949")
    m = next(m for m in zf.infolist() if nm(m).endswith("2106.csv"))
    n = 0
    with zf.open(m) as fh:
        txt = io.TextIOWrapper(fh, encoding="cp949", errors="replace")
        for ch in pd.read_csv(txt, usecols=["대여일시"], dtype=str, chunksize=500_000):
            n += ch["대여일시"].str.startswith("2021-06-15 08").sum()
print("파일 2021-06-15 08시:", n, "/ API: 2,538")

파일 2021-06-15 08시: 2437 / API: 2,538


In [40]:
import datetime as dt
r = requests.get("http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getVilageFcst",
    params=dict(serviceKey=os.environ["DATA_GO_KR_KEY"], dataType="JSON", numOfRows=10, pageNo=1,
                base_date=dt.date.today().strftime("%Y%m%d"), base_time="0500", nx=60, ny=127), timeout=30)
print("단기예보:", r.json()["response"]["header"])

r = requests.get("http://apis.data.go.kr/B090041/openapi/service/SpcdeInfoService/getRestDeInfo",
    params=dict(serviceKey=os.environ["DATA_GO_KR_KEY"], solYear="2026", solMonth="10", _type="json"), timeout=30)
print("특일:", r.text[:300])

단기예보: {'resultCode': '00', 'resultMsg': 'NORMAL_SERVICE'}
특일: {"response":{"header":{"resultCode":"00","resultMsg":"NORMAL SERVICE."},"body":{"items":{"item":[{"dateKind":"01","dateName":"개천절","isHoliday":"Y","locdate":20261003,"seq":1},{"dateKind":"01","dateName":"대체공휴일(개천절)","isHoliday":"Y","locdate":20261005,"seq":1},{"dateKind":"01","dateName":"한글날","isHol
